# W09: 시즌 프로모션 기획

시즌을 선택하고 이벤트 5종을 Gemini로 기획안 생성 후 우선순위로 정리합니다.


In [ ]:
import io
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for fp in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(fp)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}
    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        result = model.generate_content(prompt)
        return getattr(result, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

season = "여름"
uploaded = safe_upload()
if uploaded:
    print("첨부 파일이 있으면 참고용으로 사용합니다.")

prompt = (
    f"{season} 시즌을 위한 FnB 홍보 이벤트 5종을 제시해줘. "
    "각 이벤트는 이름/혜택/타깃/예상효과 형식으로 JSON 리스트 형태로 출력"
)
fallback = '[{"이벤트":"한정세일","혜택":"메뉴 할인","타깃":"전체","예상효과":"단기 주문증가"}, {"이벤트":"신메뉴체험","혜택":"시식혜택","타깃":"신규","예상효과":"전환 증가"}, {"이벤트":"재구매쿠폰","혜택":"다음 주문 할인","타깃":"재방문","예상효과":"재구매율 상승"}, {"이벤트":"세트추천","혜택":"번들할인","타깃":"중간가격층","예상효과":"객단가 상승"}, {"이벤트":"시간대 할인","혜택":"오픈런","타깃":"저조시간","예상효과":"공백시간 매출 개선"}]'

raw = use_gemini(prompt, fallback)
print(raw)
